# Transformar la data de Sprints
1. Leer la tabla de bronze `sprints`
1. Mantener solo las colunas requeridas para analítica (Drop `url` column)
1. Estandarizar los nombres de la columnas a  snake_case y traducir a español (`constructorId` → `id_constructor`, `driverId` → `id_piloto`, `raceName` → `nombre_carrera`, `positionText` → `posicion_final_texto`)
1. Renombrar las columnas a  un nombre más significativo y traducir a español  (`date` → `fecha_carrera`, `grid` → `posicion_grilla`, `laps` → `vueltas_completadas`, `number` → `numero_de tarjeta`, `position` → `posicion_final` , `season` -> `temporada`,  `round` -> `ronda`)
1. Excluir registros donde `temporada`, `ronda`, `id_constrcutor` o `id_piloto` sean nulos (validación clave del negocio)
1. Remover registros duplicados
1. Transformar valores de la columna `nombre carrera` a Title Case
1. Escribir la data transformada a la tabla silver `springs`


#### 


In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
bronze_table = f"{catalogo}.{esquema_bronze}.sprints"
silver_table = f"{catalogo}.{esquema_silver}.sprints"

In [0]:
from pyspark.sql import functions as F

#### 1 al 4 Leer la tabla de bronze, mantener solo las colunas requeridas y estandarizar los nombres de la columnas

In [0]:
sprints_df = (
  spark.table(bronze_table)
       .select("season",
              "round",
              "constructorId",
              "driverId",
              "date",
              "raceName",
              "grid",
              "laps",
              "number",
              "points",
              "position",
              "positionText",
              "status",
              "ingesta_timestamp",
              "archivo_fuente")
       .withColumnsRenamed({
            "constructorId": "id_constructor",
            "date":"fecha",
            "status": "estado",
            "season": "temporada",
            "round": "numero_carrera",
            "driverId": "id_piloto",
            "raceName": "nombre_carrera",
            "date": "fecha_carrera",
            "grid": "posicion_grilla",
            "laps": "vueltas_completadas",
            "number": "numero_tarjeta",
            "points": "puntos",
            "position": "posicion_final",
            "positionText": "posicion_final_texto"
        })
)

#### 5 & 6 Validaciones de calidad de datos
- Excluir registros donde `temporada`, `ronda`, `id_constrcutor` o `id_piloto` sean nulos (validación clave del negocio)
- Remover registros duplicados

In [0]:
sprints_valid_df = (
    sprints_df
        .filter(
            F.col("temporada").isNotNull() &
            F.col("numero_carrera").isNotNull() &
            F.col("id_constructor").isNotNull() &
            F.col("id_piloto").isNotNull() 
        )
        .dropDuplicates(["temporada", "numero_carrera", "id_constructor", "id_piloto"])
)

In [0]:
display(sprints_df.count() - sprints_valid_df.count())

#### 7 - Transformar valores de la columna `nombre carrera` a Title Case

In [0]:
from pyspark.sql.functions import when, col
sprints_final_df = (
    sprints_valid_df
        .withColumn('nombre_carrera', F.initcap(F.col("nombre_carrera")))
        .withColumn('estado',
                    when(col("estado").contains("Disqualified"), "Disqualificado")
                    .when(col("estado").contains("Accident"), "Accidente")
                    .when(col("estado").contains("Collision"), "Colisión")
                    .otherwise(col("estado")))
        )


In [0]:
display(sprints_final_df)

#### 8 - Escribir la data transformada a la tabla silver `springs`

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))